# Cross-Regional Events Co-occurrence: All 16 Rats, First Hour of Postsleep

## Setup

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# lab repo root on the path, so `modules` imports work from this folder
REPO_ROOT = next(
    p
    for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "modules" / "project_config.py").exists()
)
sys.path.insert(0, str(REPO_ROOT))
from modules.project_config import get_path  # noqa: E402

# local paths come from config.yaml (see config.example.yaml)
ROOT = get_path("R1_8_root")  # holds Rat_HM_Ephys_TD_Analysis_R1_8 and _R9_16
assert get_path("R9_16_root") == ROOT, "these notebooks expect R1_8_root == R9_16_root"
WAVE_CACHE = get_path("event_cache_root")
SESSION_MAP_CSV = get_path("session_map_csv")
DAY_GRID_XLSX = get_path("day_grid_xlsx")
OUT = REPO_ROOT / "results" / "leticia_event_coupling"  # figures and caches
SPINDLE_METHOD = "wavelet_amp_1_ampcore_3"
PHASES = ["postsleep"]

BOUT_KEYS = ["rat", "date", "trial", "nrem_bout_index"]
AREAS = ["PL", "RSC", "HPC"]
RATS = list(range(1, 17))  # all 16 rats, not the 7-rat subset with trusted ripples

N_ITER = 1000
SHIFT_RATS = RATS

FIGDIR = OUT / "cross_regional_16rats_first_hour_figures"
FIGDIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name):
    fig.savefig(FIGDIR / f"{name}.png", dpi=200, bbox_inches="tight")
    print("saved:", name)


print("drive reachable:", ROOT.exists(), " (False is fine - we load from WAVE_CACHE)")
print("rats:", RATS, "| circular shift on:", SHIFT_RATS, "|", N_ITER, "iterations")

## Load

In [ ]:
from modules.event_coupling.build_tables import add_condition, build_wave

ripples = add_condition(
    build_wave(ROOT, "ripple", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)
deltas = add_condition(
    build_wave(ROOT, "delta", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)
spindles = add_condition(
    build_wave(ROOT, "spindle", SPINDLE_METHOD, PHASES, WAVE_CACHE, rebuild=False),
    SESSION_MAP_CSV,
)

rip = ripples[(ripples["rat"].isin(RATS)) & (ripples["group"] != "other")]
dlt = deltas[
    (deltas["rat"].isin(RATS))
    & (deltas["group"] != "other")
    & (deltas["area"].isin(AREAS))
]
spn = spindles[
    (spindles["rat"].isin(RATS))
    & (spindles["group"] != "other")
    & (spindles["area"].isin(AREAS))
]

print(f"{len(rip):,} ripples | {len(dlt):,} deltas | {len(spn):,} spindles")
print(rip.groupby("group").size().to_string())

### Confirm  we have excluded the "bad" days

In [ ]:
from modules.event_coupling.excluded_days import excluded_table

ex = excluded_table()

for name, df in [("ripples", rip), ("deltas", dlt), ("spindles", spn)]:
    check = ex.merge(
        df[["rat", "date"]]
        .assign(date=pd.to_datetime(df["date"]).dt.normalize())
        .drop_duplicates(),
        on=["rat", "date"],
    )
    print(
        f"{name}: {len(ex)} rat-days on the exclusion list | "
        f"still present: {len(check)} (want 0)"
    )
    assert len(check) == 0, (
        f"exclusion did not fire for {name} - is excluded_days.py in this folder?"
    )

## Add Time labels

In [ ]:
from modules.event_coupling.gl_sessions import add_gl_sessions, build_gl_sessions
from modules.event_coupling.hc_sessions import add_session_type, build_day_table


def prep_keys(df):
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["rat"] = df["rat"].astype("int64")
    return df


gl = prep_keys(build_gl_sessions(ROOT, cache_dir=WAVE_CACHE, rebuild=False))
days = add_session_type(build_day_table(DAY_GRID_XLSX, verbose=False))


def add_context(df):
    df = add_gl_sessions(prep_keys(df), gl)
    return df.merge(
        prep_keys(days)[["rat", "date", "session_type"]].drop_duplicates(),
        on=["rat", "date"],
        how="left",
    )


rip_ctx = add_context(rip)
dlt_ctx = add_context(dlt)

first = rip_ctx.groupby("rat")["date"].transform("min")
rip_ctx["days_elapsed"] = (rip_ctx["date"] - first).dt.days
dlt_ctx = dlt_ctx.merge(
    rip_ctx[["rat", "date", "days_elapsed"]].drop_duplicates(),
    on=["rat", "date"],
    how="left",
)

print(
    rip_ctx.groupby(["rat", "group"])["date"]
    .nunique()
    .unstack(fill_value=0)
    .to_string()
)

## Build missing spindle context

In [ ]:
def add_spindle_context(df):
    df = add_gl_sessions(prep_keys(df), gl)
    df = df.merge(
        prep_keys(days)[["rat", "date", "session_type"]].drop_duplicates(),
        on=["rat", "date"],
        how="left",
    )
    return df


spn_ctx = add_spindle_context(spn)
spn_ctx = spn_ctx.merge(
    rip_ctx[["rat", "date", "days_elapsed"]].drop_duplicates(),
    on=["rat", "date"],
    how="left",
)

PAIRS = [("PL", "RSC"), ("PL", "HPC"), ("RSC", "HPC")]
print(f"{len(spn_ctx):,} spindle rows with context")
print("pairs:", PAIRS)

## Check: is the first hour over-represented?

I made this after finding out the first hour was being over-represented previously.
This check is mainly to plot how much NREM sleep occured per hour.
Is to make sure the previous mistake is not happening.

In [ ]:
TRIAL_LEN = 1_800_000  # 30 min at 1000Hz.


def add_elapsed_time(df, index_col):
    """True elapsed time since the start of postsleep, correcting for
    peak_index resetting to 0 at the start of EACH trial file."""
    min_trial = df.groupby(["rat", "date"])["trial"].transform("min")
    return (df["trial"] - min_trial) * TRIAL_LEN + df[index_col]


def split_bout_by_hour(start, end):
    """Minutes of NREM in each of hours 1-4 for one bout [start, end)."""
    HOUR = 60 * 60 * 1000
    out = np.zeros(4)
    for h in range(4):
        lo, hi = h * HOUR, (h + 1) * HOUR
        overlap = max(0, min(end, hi) - max(start, lo))
        out[h] = overlap / 1000 / 60
    return out


bouts_check = rip_ctx[
    ["rat", "date", "trial", "nrem_bout_start_index", "nrem_bout_end_index"]
].drop_duplicates()
bouts_check["elapsed_start"] = add_elapsed_time(bouts_check, "nrem_bout_start_index")
bouts_check["elapsed_end"] = add_elapsed_time(bouts_check, "nrem_bout_end_index")

hour_mins = np.array(
    [
        split_bout_by_hour(s, e)
        for s, e in zip(bouts_check["elapsed_start"], bouts_check["elapsed_end"])
    ]
)
bouts_check[["hour1", "hour2", "hour3", "hour4"]] = hour_mins
per_rat_hours = bouts_check.groupby("rat")[["hour1", "hour2", "hour3", "hour4"]].sum()

fig, ax = plt.subplots(figsize=(10, 5))
bottom = np.zeros(len(per_rat_hours))
colors = ["#2c5f8a", "#5b9bd5", "#a9c9e8", "#d6e4f0"]
for i, hour_col in enumerate(["hour1", "hour2", "hour3", "hour4"]):
    ax.bar(
        per_rat_hours.index.astype(str),
        per_rat_hours[hour_col],
        bottom=bottom,
        label=f"Hour {i + 1}" if i < 3 else "Last hour",
        color=colors[i],
    )
    bottom += per_rat_hours[hour_col].values
ax.set_xlabel("Rat")
ax.set_ylabel("NREM duration (minutes)")
ax.set_title("Postsleep NREM duration by rat and recording hour (all 16 rats)")
ax.legend(title="Recording hour")
plt.tight_layout()
plt.show()

## Restrict to the first hour of postsleep

In [ ]:
ONE_HOUR_SAMPLES = 60 * 60 * 1000  # 1000 Hz -> 1 sample index = 1 ms


def filter_first_hour_fixed(df, index_col):
    """Correct version: filters on true elapsed time (trial position x 30
    min + within-trial index), not raw peak_index. See the verification
    bar chart above and cross_regional_first_hour.ipynb for how this bug
    was found and validated."""
    df = df.copy()
    df["_elapsed"] = add_elapsed_time(df, index_col)
    return df[df["_elapsed"] < ONE_HOUR_SAMPLES].drop(columns="_elapsed")


dlt_ctx_full, spn_ctx_full = dlt_ctx, spn_ctx
dlt_ctx = filter_first_hour_fixed(dlt_ctx_full, "delta_peak_index")
spn_ctx = filter_first_hour_fixed(spn_ctx_full, "spindle_peak_index")

for name, full, hr in [
    ("deltas", dlt_ctx_full, dlt_ctx),
    ("spindles", spn_ctx_full, spn_ctx),
]:
    print(
        f"{name}: {len(full):,} total -> {len(hr):,} in true first hour "
        f"({100 * len(hr) / len(full):.1f}%)"
    )

## Self-tests

In [ ]:
def count_events_with_partner(seed, target, lag_lo, lag_hi, seed_col, target_col):
    """Unique seed-region events with a target-region event in the given lag
    window, per rat-day. lag = seed_peak - target_peak."""
    rows = []
    for (rat, date, trial), sg in seed.groupby(["rat", "date", "trial"]):
        tg = target[
            (target["rat"] == rat)
            & (target["date"] == date)
            & (target["trial"] == trial)
        ]
        n_coupled = 0
        for bout in sg["nrem_bout_index"].unique():
            s_b = sg.loc[sg["nrem_bout_index"] == bout, seed_col].values
            t_b = (
                tg.loc[tg["nrem_bout_index"] == bout, target_col].values
                if not tg.empty
                else np.array([])
            )
            if len(s_b) == 0 or len(t_b) == 0:
                continue
            lags = s_b[:, None] - t_b[None, :]
            has_partner = ((lags >= lag_lo) & (lags < lag_hi)).any(axis=1)
            n_coupled += int(has_partner.sum())
        rows.append(
            {
                "rat": rat,
                "date": date,
                "trial": trial,
                "n_coupled": n_coupled,
                "n_seed": len(sg),
            }
        )
    return pd.DataFrame(rows)


def _synth(n_days=3, seed=0):
    rng = np.random.default_rng(seed)
    D, R = [], []
    for day in range(1, n_days + 1):
        d = np.sort(rng.uniform(0, 20000, 60))
        r = np.sort(rng.uniform(0, 20000, 10))
        if day == 2:
            take = rng.choice(len(d), 4, replace=False)
            r = np.sort(np.concatenate([r, d[take] + 140]))
        r = r[(r > 0) & (r < 20000)]
        D.append(
            pd.DataFrame(
                {
                    "rat": 1,
                    "date": f"day{day}",
                    "trial": 1,
                    "nrem_bout_index": 0,
                    "wave_peak_index": d,
                }
            )
        )
        R.append(
            pd.DataFrame(
                {
                    "rat": 1,
                    "date": f"day{day}",
                    "trial": 1,
                    "nrem_bout_index": 0,
                    "wave_peak_index": r,
                }
            )
        )
    return pd.concat(D, ignore_index=True), pd.concat(R, ignore_index=True)


_d, _r = _synth()
_t = count_events_with_partner(_r, _d, -250, -50, "wave_peak_index", "wave_peak_index")
assert _t.loc[_t.date == "day2", "n_coupled"].iloc[0] >= 4
print("per-rat-day partner-counting self-test PASSED")

## [SILENCED] Circular-shift method 

**We are not using it.** 
We computed a lot of artefacts from this maethod
For this analysis, 16 rats, we kept to the permutation test and the over-time plots.

In [ ]:
# import pickle

# CACHE_XREGIONAL = OUT / "significance_cache_crossregional_16rats"; CACHE_XREGIONAL.mkdir(parents=True, exist_ok=True)
# N_ITER_SIG = 1000
# CHECKPOINT_EVERY = 50
# edges = np.arange(-1000, 1020, 20)
# left = edges[:-1]


# def windowed_hist(s_b, t_b, lo, hi, edges):
#     lo_idx = np.searchsorted(t_b, s_b + lo, side="left")
#     hi_idx = np.searchsorted(t_b, s_b + hi, side="right")
#     counts = hi_idx - lo_idx
#     total = counts.sum()
#     if total == 0:
#         return np.zeros(len(edges) - 1)
#     seed_idx_repeated = np.repeat(np.arange(len(s_b)), counts)
#     group_offsets = np.repeat(np.cumsum(counts) - counts, counts)
#     offset_within_group = np.arange(total) - group_offsets
#     j_indices = np.repeat(lo_idx, counts) + offset_within_group
#     lags_flat = s_b[seed_idx_repeated] - t_b[j_indices]
#     return np.histogram(lags_flat, bins=edges)[0]


# def cross_regional_dist(seed_df, target_df, area_seed, area_target, wave,
#                         n_iter=N_ITER_SIG, seed=0, checkpoint_every=CHECKPOINT_EVERY):
#     """1000-iteration circular-shift surrogate, switched off and not used in
#     this notebook. See the markdown above for why."""
#     raise NotImplementedError("silenced -- not used in this 16-rat notebook")

## [SILENCED] Wave similarity

Not used in this notebook.
Wave similarity (Pearson + cross-correlation between co-occurring waveforms, compared against a control) is not part of this 16-rat analysis 

Very time consuming for this analysis but we have it here in case it's wanted later

In [ ]:
# # wave-similarity code: see cross_regional_full.ipynb / cross_regional_first_hour.ipynb
# raise NotImplementedError("silenced -- not used in this 16-rat notebook")

## Permutation test: does co-occurrence change over time?

### NREM-duration day cutoff
**Rule:** at leat 10 minute sof NREM sleep 

In [ ]:
MIN_NREM_MIN = 10


bout_dur = dlt_ctx_full[
    ["rat", "date", "trial", "nrem_bout_start_index", "nrem_bout_end_index"]
].drop_duplicates()
bout_dur["elapsed_start"] = add_elapsed_time(bout_dur, "nrem_bout_start_index")
bout_dur["elapsed_end"] = add_elapsed_time(bout_dur, "nrem_bout_end_index")
bout_dur["hour1_min"] = [
    split_bout_by_hour(s, e)[0]
    for s, e in zip(bout_dur["elapsed_start"], bout_dur["elapsed_end"])
]
day_nrem_hour1 = bout_dur.groupby(["rat", "date"])["hour1_min"].sum()

good_days = day_nrem_hour1[day_nrem_hour1 >= MIN_NREM_MIN].reset_index()[
    ["rat", "date"]
]
print(
    f"rat-days: {day_nrem_hour1.shape[0]} total -> {len(good_days)} with >= {MIN_NREM_MIN} min "
    f"NREM in hour 1 ({day_nrem_hour1.shape[0] - len(good_days)} dropped)"
)

In [ ]:
def compute_cutoff_dropped(seed_df, target_df, area_seed, area_target, wave, window_ms):

    peak_col = f"{wave}_peak_index"
    lo, hi = window_ms
    s = seed_df[seed_df["area"] == area_seed]
    t = target_df[target_df["area"] == area_target]
    good_set = set(zip(good_days["rat"], good_days["date"]))
    rows = []
    for (rat, date), sg in s.groupby(["rat", "date"]):
        if (rat, date) in good_set:
            continue
        nrem_min = day_nrem_hour1.get((rat, date), float("nan"))
        if not (nrem_min < MIN_NREM_MIN):
            continue  # not dropped by the cutoff (e.g. 0 seed events), handled elsewhere
        tg = t[(t["rat"] == rat) & (t["date"] == date)]
        n_coupled = 0
        for trial in sg["trial"].unique():
            sg_t, tg_t = sg[sg["trial"] == trial], tg[tg["trial"] == trial]
            for bout in sg_t["nrem_bout_index"].unique():
                s_b = sg_t.loc[sg_t["nrem_bout_index"] == bout, peak_col].values
                t_b = (
                    tg_t.loc[tg_t["nrem_bout_index"] == bout, peak_col].values
                    if not tg_t.empty
                    else np.array([])
                )
                if len(s_b) == 0 or len(t_b) == 0:
                    continue
                lags = s_b[:, None] - t_b[None, :]
                n_coupled += int(((lags >= lo) & (lags < hi)).any(axis=1).sum())
        if len(sg) == 0:
            continue
        row = rip_ctx[(rip_ctx["rat"] == rat) & (rip_ctx["date"] == date)]
        if row.empty:
            continue
        row = row.iloc[0]
        rows.append(
            {
                "rat": rat,
                "date": date,
                "n_coupled": n_coupled,
                "n_seed": len(sg),
                "group": row["group"],
                "days_elapsed": row["days_elapsed"],
                "gl_num": row.get("gl_num"),
                "session_num": row.get("session_num"),
                "nrem_min_hour1": nrem_min,
            }
        )
    df = pd.DataFrame(rows)
    if not df.empty:
        df["ratio_proxy"] = df["n_coupled"] / df["n_seed"].replace(0, np.nan)
    return df

In [ ]:
def per_day_cooccurrence(seed_df, target_df, area_seed, area_target, wave, window_ms):

    peak_col = f"{wave}_peak_index"
    lo, hi = window_ms
    s = seed_df[seed_df["area"] == area_seed]
    t = target_df[target_df["area"] == area_target]
    rows = []
    for (rat, date), sg in s.groupby(["rat", "date"]):
        tg = t[(t["rat"] == rat) & (t["date"] == date)]
        n_coupled = 0
        for trial in sg["trial"].unique():
            sg_t, tg_t = sg[sg["trial"] == trial], tg[tg["trial"] == trial]
            for bout in sg_t["nrem_bout_index"].unique():
                s_b = sg_t.loc[sg_t["nrem_bout_index"] == bout, peak_col].values
                t_b = (
                    tg_t.loc[tg_t["nrem_bout_index"] == bout, peak_col].values
                    if not tg_t.empty
                    else np.array([])
                )
                if len(s_b) == 0 or len(t_b) == 0:
                    continue
                lags = s_b[:, None] - t_b[None, :]
                n_coupled += int(((lags >= lo) & (lags < hi)).any(axis=1).sum())
        if len(sg) == 0:
            continue
        row = rip_ctx[(rip_ctx["rat"] == rat) & (rip_ctx["date"] == date)]
        if row.empty:
            continue
        row = row.iloc[0]
        rows.append(
            {
                "rat": rat,
                "date": date,
                "n_coupled": n_coupled,
                "n_seed": len(sg),
                "group": row["group"],
                "days_elapsed": row["days_elapsed"],
                "gl_num": row.get("gl_num"),
                "session_num": row.get("session_num"),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    return df.merge(good_days, on=["rat", "date"])

In [ ]:
COHORTS = {
    "build-up (rats 1-4)": [r for r in RATS if 1 <= r <= 4],
    "build-up (rats 5-8)": [r for r in RATS if 5 <= r <= 8],
    "update (rats 9-12)": [r for r in RATS if 9 <= r <= 12],
    "update (rats 13-16)": [r for r in RATS if 13 <= r <= 16],
}

In [ ]:
def find_missing_points(daily_df, rats_to_check, group="Training"):

    expected = rip_ctx[
        (rip_ctx["rat"].isin(rats_to_check)) & (rip_ctx["group"] == group)
    ]
    if group == "Training":
        expected = expected.dropna(subset=["gl_num", "session_num"])
        cols = ["rat", "date", "gl_num", "session_num"]
    else:
        cols = ["rat", "date"]
    expected = expected[cols].drop_duplicates()
    present = set(zip(daily_df["rat"], daily_df["date"]))
    good_set = set(zip(good_days["rat"], good_days["date"]))
    notes = []
    for _, row in expected.iterrows():
        key = (row["rat"], row["date"])
        if key in present:
            continue
        date_str = row["date"].date() if hasattr(row["date"], "date") else row["date"]
        if key not in good_set:
            nrem_min = day_nrem_hour1.get(key, float("nan"))
            reason = (
                f"below hour-1 NREM cutoff ({nrem_min:.1f} min < {MIN_NREM_MIN} min)"
            )
        else:
            reason = "0 seed events that day"
        if group == "Training":
            notes.append(
                f"Rat {row['rat']}: {date_str} (GL{int(row['gl_num'])}_S{int(row['session_num'])}) "
                f"missing -- {reason}"
            )
        else:
            notes.append(f"Rat {row['rat']}: {date_str} (HC) missing -- {reason}")
    return notes


def spearman_permutation_plot(
    daily_df, area_seed, area_target, wave, ylim=None, dropped_df=None
):

    from scipy.stats import spearmanr

    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(daily_df, all_rats_in_cohorts, group="Training")
    gap_notes = []

    sub_all = daily_df.copy()
    sub_all["ratio_proxy"] = sub_all["n_coupled"] / sub_all["n_seed"].replace(0, np.nan)
    sub_all = sub_all.dropna(subset=["gl_num", "session_num", "ratio_proxy"])
    if sub_all.empty:
        print(
            f"{area_seed}-{area_target} {wave}: no GL/session-labelled data, skipping"
        )
        return []

    drop_all = pd.DataFrame()
    if dropped_df is not None and not dropped_df.empty:
        drop_all = (
            dropped_df[dropped_df["group"] == "Training"]
            .dropna(subset=["gl_num", "session_num"])
            .copy()
        )

    rng = np.random.default_rng(0)
    results = []
    fig, axes = plt.subplots(1, 4, figsize=(26, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        dsub = (
            drop_all[drop_all["rat"].isin(rats)].copy()
            if not drop_all.empty
            else pd.DataFrame()
        )
        if sub.empty and dsub.empty:
            ax.set_title(label, fontsize=11)
            continue

        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        if not dsub.empty:
            dsub["step"] = dsub["gl_num"] * 10 + dsub["session_num"]
        order = sorted(
            set(sub["step"].unique())
            | (set(dsub["step"].unique()) if not dsub.empty else set())
        )
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            if len(r) < 5:
                if len(r) > 0:
                    gap_notes.append(
                        f"rat {rat}: excluded -- only {len(r)} qualifying day(s), need >=5"
                    )
                continue
            rho, _ = spearmanr(r["days_elapsed"], r["ratio_proxy"])
            perm_rhos = np.array(
                [
                    spearmanr(r["days_elapsed"], rng.permutation(r["ratio_proxy"]))[0]
                    for _ in range(1000)
                ]
            )
            p_perm = (np.sum(np.abs(perm_rhos) >= abs(rho)) + 1) / 1001
            z_perm = (rho - perm_rhos.mean()) / (perm_rhos.std() + 1e-9)

            results.append(
                {
                    "pair": f"{area_seed}-{area_target}",
                    "wave": wave,
                    "cohort": label,
                    "rat": rat,
                    "n_days": len(r),
                    "rho": rho,
                    "z_perm": z_perm,
                    "p_perm": p_perm,
                }
            )

            g = r.groupby("step")["ratio_proxy"].mean().reindex(order)
            xs = [pos[s] for s in order]
            ax.plot(
                xs,
                g.values,
                marker="o",
                ms=4,
                lw=1.2,
                alpha=0.8,
                label=f"rat {rat}  \u03c1={rho:+.2f}{'*' if p_perm < 0.05 else ''}",
            )

            missing_idx = sorted(pos[s] for s in order if s not in set(r["step"]))
            if missing_idx:
                ranges = []
                start = prev = missing_idx[0]
                for idx in missing_idx[1:]:
                    if idx == prev + 1:
                        prev = idx
                        continue
                    ranges.append((start, prev))
                    start = prev = idx
                ranges.append((start, prev))

                def _lab(s):
                    return f"GL{int(s // 10)}_S{int(s % 10)}"

                parts = [
                    _lab(order[a])
                    if a == b
                    else f"{_lab(order[a])}\u2013{_lab(order[b])}"
                    for a, b in ranges
                ]
                gap_notes.append(f"rat {rat}: no " + ", ".join(parts))

        if not dsub.empty:
            xs_d = [pos[s] for s in dsub["step"]]
            ax.scatter(
                xs_d,
                dsub["ratio_proxy"],
                marker="o",
                s=26,
                facecolors="none",
                edgecolors="0.5",
                linewidths=1.1,
                zorder=5,
                label="cutoff-dropped (raw)" if ax is axes[0] else None,
            )

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.2, lw=0.5, axis="y")
        if ylim is not None:
            ax.set_ylim(ylim)

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area_seed}\u2013{area_target}  {wave}  co-occurrence over time, Training days (first hour)   "
        "(shading marks goal-location blocks; * = permutation p < 0.05; "
        "hollow circles = cutoff-dropped, shown raw)",
        fontsize=12,
    )

    plt.tight_layout()

    n_lines = int(bool(missing_notes)) + int(bool(gap_notes))
    if n_lines:
        fig.subplots_adjust(bottom=0.22 + 0.13 * n_lines)
        y = 0.02
        if missing_notes:
            fig.text(
                0.5,
                y,
                "Missing point(s): " + " | ".join(missing_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="firebrick",
                wrap=True,
            )
            y += 0.12
        if gap_notes:
            fig.text(
                0.5,
                y,
                "Line gap(s) -- rat did not run that session, not missing data: "
                + " | ".join(gap_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="0.35",
                wrap=True,
            )

    save_fig(fig, f"xregional_overtime_{wave}_{area_seed}_{area_target}")
    plt.show()
    return results

### Raw ratio, Home-cage days

In [ ]:
def hc_overtime_plot(
    daily_df, area_seed, area_target, wave, ylim=None, dropped_df=None
):

    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(daily_df, all_rats_in_cohorts, group="HC")
    excluded_notes = []

    sub_all = daily_df[daily_df["group"] == "HC"].copy()
    sub_all["ratio_proxy"] = sub_all["n_coupled"] / sub_all["n_seed"].replace(0, np.nan)
    sub_all = sub_all.dropna(subset=["ratio_proxy"])

    drop_all = pd.DataFrame()
    if dropped_df is not None and not dropped_df.empty:
        drop_all = dropped_df[dropped_df["group"] == "HC"].copy()

    if sub_all.empty and drop_all.empty:
        print(f"{area_seed}-{area_target} {wave}: no HC data, skipping")
        return

    fig, axes = plt.subplots(1, 4, figsize=(26, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = sub_all[sub_all["rat"].isin(rats)].copy()
        dsub = (
            drop_all[drop_all["rat"].isin(rats)].copy()
            if not drop_all.empty
            else pd.DataFrame()
        )
        if sub.empty and dsub.empty:
            ax.set_title(label, fontsize=11)
            continue
        for rat in rats:
            r = sub[sub["rat"] == rat].sort_values("days_elapsed")
            if len(r) < 5:
                if len(r) > 0:
                    excluded_notes.append(
                        f"rat {rat}: excluded -- only {len(r)} qualifying HC day(s), need >=5"
                    )
                continue
            ax.plot(
                r["days_elapsed"],
                r["ratio_proxy"],
                marker="o",
                ms=4,
                lw=1.2,
                alpha=0.8,
                label=f"rat {rat}",
            )
        if not dsub.empty:
            ax.scatter(
                dsub["days_elapsed"],
                dsub["ratio_proxy"],
                marker="o",
                s=26,
                facecolors="none",
                edgecolors="0.5",
                linewidths=1.1,
                zorder=5,
                label="cutoff-dropped (raw)" if ax is axes[0] else None,
            )
        ax.set_title(label, fontsize=11)
        ax.set_xlabel("days elapsed")
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.2, lw=0.5, axis="y")
        if ylim is not None:
            ax.set_ylim(ylim)

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area_seed}\u2013{area_target}  {wave}  co-occurrence over time, Home-cage days (raw, first hour)   "
        "(hollow circles = cutoff-dropped, shown raw)",
        fontsize=12,
    )

    plt.tight_layout()

    n_lines = int(bool(missing_notes)) + int(bool(excluded_notes))
    if n_lines:
        fig.subplots_adjust(bottom=0.22 + 0.13 * n_lines)
        y = 0.02
        if missing_notes:
            fig.text(
                0.5,
                y,
                "Missing point(s): " + " | ".join(missing_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="firebrick",
                wrap=True,
            )
            y += 0.12
        if excluded_notes:
            fig.text(
                0.5,
                y,
                "Excluded rat(s) -- fewer than 5 qualifying HC days: "
                + " | ".join(excluded_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="0.35",
                wrap=True,
            )

    save_fig(fig, f"xregional_hc_raw_{wave}_{area_seed}_{area_target}")
    plt.show()

In [ ]:
WINDOW_MS = (-750, 750)

xregional_overtime_results = []
all_cutoff_dropped = []
for area_seed, area_target in PAIRS:
    for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
        daily_df = per_day_cooccurrence(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        dropped_df = compute_cutoff_dropped(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        all_cutoff_dropped.append((area_seed, area_target, wave, dropped_df))

        if daily_df.empty:
            print(f"{area_seed}-{area_target} {wave}: no data, skipping")
            continue

        ratio_all = pd.concat(
            [
                daily_df["n_coupled"] / daily_df["n_seed"].replace(0, np.nan),
                dropped_df["ratio_proxy"]
                if not dropped_df.empty
                else pd.Series(dtype=float),
            ]
        ).dropna()
        shared_ylim = (0, ratio_all.max() * 1.08) if not ratio_all.empty else None

        xregional_overtime_results.extend(
            spearman_permutation_plot(
                daily_df,
                area_seed,
                area_target,
                wave,
                ylim=shared_ylim,
                dropped_df=dropped_df,
            )
        )
        hc_overtime_plot(
            daily_df,
            area_seed,
            area_target,
            wave,
            ylim=shared_ylim,
            dropped_df=dropped_df,
        )

In [ ]:
overtime_summary = pd.DataFrame(xregional_overtime_results)
overtime_summary["significant"] = overtime_summary["p_perm"] < 0.05

print("=== per-rat Spearman correlation, co-occurrence ratio vs. time ===")
print("(Training days, first hour of postsleep, raw/un-normalized ratio_proxy;")
print("* = permutation p < 0.05)")
print(f"{len(overtime_summary)} rat-level permutation tests total\n")
print(
    overtime_summary.sort_values(["pair", "wave", "p_perm"])[
        [
            "pair",
            "wave",
            "cohort",
            "rat",
            "n_days",
            "rho",
            "z_perm",
            "p_perm",
            "significant",
        ]
    ].to_string(index=False)
)

In [ ]:
print("=== Cutoff-dropped rat-days: raw ratio_proxy vs. hour-1 NREM minutes ===")
print(f"(MIN_NREM_MIN = {MIN_NREM_MIN} -- every row below has less than this)")
rows = []
for area_seed, area_target, wave, dropped_df in all_cutoff_dropped:
    if dropped_df.empty:
        continue
    d = dropped_df.copy()
    d["pair"] = f"{area_seed}-{area_target}"
    d["wave"] = wave
    rows.append(d)

if rows:
    cutoff_table = pd.concat(rows, ignore_index=True)
    cutoff_table["date"] = cutoff_table["date"].apply(
        lambda d: d.date() if hasattr(d, "date") else d
    )
    cutoff_table = cutoff_table.sort_values(["pair", "wave", "rat", "date"])
    print(f"{len(cutoff_table)} rat-day rows total, cut across all pairs/waves\n")
    print(
        cutoff_table[
            [
                "pair",
                "wave",
                "rat",
                "date",
                "group",
                "nrem_min_hour1",
                "n_coupled",
                "n_seed",
                "ratio_proxy",
            ]
        ].to_string(index=False)
    )
else:
    print("none -- no rat-day failed the hour-1 NREM cutoff for any pair/wave")

## Over-time plots, normalized to nearest home-cage session

In [ ]:
def dropped_to_notes(dropped):
    notes = []
    for d in dropped:
        date_str = d["date"].date() if hasattr(d["date"], "date") else d["date"]
        notes.append(
            f"Rat {d['rat']}: {date_str} missing -- nearest two HC days "
            f"({d['nearest_hc_days_elapsed']}) both had 0 co-occurrences"
        )
    return notes


def nearest_hc_normalize(daily_df):
    daily_df = daily_df.copy()
    daily_df["ratio_proxy"] = daily_df["n_coupled"] / daily_df["n_seed"].replace(
        0, np.nan
    )
    out_rows = []
    dropped = []
    for rat, g in daily_df.groupby("rat"):
        hc = (
            g[g["group"] == "HC"]
            .dropna(subset=["ratio_proxy"])
            .sort_values("days_elapsed")
        )
        train = g[g["group"] == "Training"].dropna(
            subset=["ratio_proxy", "gl_num", "session_num"]
        )
        if hc.empty or train.empty:
            continue
        for _, row in train.iterrows():
            hc_sorted = hc.iloc[
                (hc["days_elapsed"] - row["days_elapsed"]).abs().argsort()
            ]
            used = None
            for rank, (_, hc_row) in enumerate(hc_sorted.iterrows()):
                if rank >= 2:
                    break
                if hc_row["ratio_proxy"] and hc_row["ratio_proxy"] > 0:
                    used = hc_row
                    break
            if used is None:
                nearest_two = hc_sorted.head(2)
                dropped.append(
                    {
                        "rat": rat,
                        "date": row["date"],
                        "days_elapsed": row["days_elapsed"],
                        "nearest_hc_days_elapsed": list(nearest_two["days_elapsed"]),
                    }
                )
                continue
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row["ratio_proxy"] / used["ratio_proxy"],
                    "nearest_hc_days_elapsed": used["days_elapsed"],
                }
            )
    return pd.DataFrame(out_rows), dropped


def plot_hc_normalized(
    norm_df,
    daily_df,
    dropped,
    area_seed,
    area_target,
    wave,
    title_suffix="nearest home-cage session",
    fname_tag="hcnorm",
):
    all_rats_in_cohorts = [r for rats in COHORTS.values() for r in rats]
    missing_notes = find_missing_points(
        daily_df, all_rats_in_cohorts, group="Training"
    ) + dropped_to_notes(dropped)
    gap_notes = []

    if norm_df.empty:
        print(f"{area_seed}-{area_target} {wave}: no data to normalize, skipping")
        return

    fig, axes = plt.subplots(1, 4, figsize=(26, 4.6), sharey=True)
    for ax, (label, rats) in zip(axes, COHORTS.items()):
        sub = norm_df[norm_df["rat"].isin(rats)].copy()
        if sub.empty:
            ax.set_title(label, fontsize=11)
            continue
        sub["step"] = sub["gl_num"] * 10 + sub["session_num"]
        order = sorted(sub["step"].unique())
        pos = {s: i for i, s in enumerate(order)}
        tick_labels = [f"GL{int(s // 10)}_S{int(s % 10)}" for s in order]

        for rat in rats:
            r = sub[sub["rat"] == rat]
            if len(r) < 3:
                if len(r) > 0:
                    gap_notes.append(
                        f"rat {rat}: excluded -- only {len(r)} qualifying normalized day(s), need >=3"
                    )
                continue
            g = r.groupby("step")["ratio"].mean().reindex(order)
            ax.plot(
                [pos[s] for s in order],
                g.values,
                lw=1.6,
                alpha=0.85,
                marker="o",
                ms=4,
                label=f"rat {rat}",
            )

            missing_idx = sorted(pos[s] for s in order if s not in set(r["step"]))
            if missing_idx:
                ranges = []
                start = prev = missing_idx[0]
                for idx in missing_idx[1:]:
                    if idx == prev + 1:
                        prev = idx
                        continue
                    ranges.append((start, prev))
                    start = prev = idx
                ranges.append((start, prev))

                def _lab(s):
                    return f"GL{int(s // 10)}_S{int(s % 10)}"

                parts = [
                    _lab(order[a])
                    if a == b
                    else f"{_lab(order[a])}\u2013{_lab(order[b])}"
                    for a, b in ranges
                ]
                gap_notes.append(f"rat {rat}: no " + ", ".join(parts))

        for i, s in enumerate(order):
            if int(s // 10) % 2 == 0:
                ax.axvspan(i - 0.5, i + 0.5, color="0.92", zorder=0)
        ax.axhline(1, color="k", lw=1.2, ls=":")
        ax.set_xticks(range(len(tick_labels)))
        ax.set_xticklabels(tick_labels, rotation=90, fontsize=7)
        ax.set_title(label, fontsize=11)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.2, lw=0.5, axis="y")

    axes[0].set_ylabel("ratio")
    fig.suptitle(
        f"{area_seed}\u2013{area_target}  {wave}  normalized to {title_suffix} (first hour)   "
        "(shading marks goal-location blocks)",
        fontsize=12,
    )

    plt.tight_layout()
    n_lines = int(bool(missing_notes)) + int(bool(gap_notes))
    if n_lines:
        fig.subplots_adjust(bottom=0.22 + 0.13 * n_lines)
        y = 0.02
        if missing_notes:
            fig.text(
                0.5,
                y,
                "Missing point(s): " + " | ".join(missing_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="firebrick",
                wrap=True,
            )
            y += 0.12
        if gap_notes:
            fig.text(
                0.5,
                y,
                "Line gap(s) -- rat did not run that session, not missing data: "
                + " | ".join(gap_notes),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="0.35",
                wrap=True,
            )

    save_fig(fig, f"xregional_{fname_tag}_{wave}_{area_seed}_{area_target}")
    plt.show()


xregional_hcnorm_results = {}
all_dropped = []
for area_seed, area_target in PAIRS:
    for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
        daily_df = per_day_cooccurrence(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        if daily_df.empty:
            print(f"{area_seed}-{area_target} {wave}: no data, skipping")
            continue
        norm_df, dropped = nearest_hc_normalize(daily_df)
        xregional_hcnorm_results[(area_seed, area_target, wave)] = norm_df
        all_dropped.append((f"{area_seed}-{area_target} {wave}", dropped))
        plot_hc_normalized(norm_df, daily_df, dropped, area_seed, area_target, wave)

print()
print(
    "=== Points removed (nearest-HC): both nearest and second-nearest HC days were 0 ==="
)
any_dropped = False
for name, dropped in all_dropped:
    for d in dropped:
        any_dropped = True
        print(
            f"  [{name}] rat {d['rat']}, {d['date']} (days_elapsed={d['days_elapsed']}): "
            f"nearest two HC days ({d['nearest_hc_days_elapsed']}) both had 0 co-occurrences"
        )
if not any_dropped:
    print("  none -- every training day found a usable HC baseline within two tries")

## Over-time plots, normalized to average home-cage session

In [ ]:
def average_hc_normalize(daily_df):

    daily_df = daily_df.copy()
    daily_df["ratio_proxy"] = daily_df["n_coupled"] / daily_df["n_seed"].replace(
        0, np.nan
    )
    out_rows = []
    no_baseline = []
    for rat, g in daily_df.groupby("rat"):
        hc = g[g["group"] == "HC"].dropna(subset=["ratio_proxy"])
        hc_nonzero = hc[hc["ratio_proxy"] > 0]
        train = g[g["group"] == "Training"].dropna(
            subset=["ratio_proxy", "gl_num", "session_num"]
        )
        if train.empty:
            continue
        if hc_nonzero.empty:
            no_baseline.append({"rat": rat, "n_hc_days": len(hc), "n_hc_nonzero": 0})
            continue
        avg_hc = hc_nonzero["ratio_proxy"].mean()
        for _, row in train.iterrows():
            out_rows.append(
                {
                    **row.to_dict(),
                    "ratio": row["ratio_proxy"] / avg_hc,
                    "avg_hc_value": avg_hc,
                    "n_hc_days_used": len(hc_nonzero),
                }
            )
    return pd.DataFrame(out_rows), no_baseline


xregional_avghcnorm_results = {}
all_no_baseline = []
for area_seed, area_target in PAIRS:
    for wave, ctx in [("delta", dlt_ctx), ("spindle", spn_ctx)]:
        daily_df = per_day_cooccurrence(
            ctx, ctx, area_seed, area_target, wave, WINDOW_MS
        )
        if daily_df.empty:
            continue
        norm_df, no_baseline = average_hc_normalize(daily_df)
        xregional_avghcnorm_results[(area_seed, area_target, wave)] = norm_df
        all_no_baseline.append((f"{area_seed}-{area_target} {wave}", no_baseline))
        # reuse plot_hc_normalized; footnote notes built here (no_baseline has no nearest_hc_days_elapsed)
        missing_for_plot = []
        for nb in no_baseline:
            missing_for_plot.append(
                {
                    "rat": nb["rat"],
                    "date": "(all training days)",
                    "nearest_hc_days_elapsed": "no usable HC day at all",
                }
            )
        plot_hc_normalized(
            norm_df,
            daily_df,
            missing_for_plot,
            area_seed,
            area_target,
            wave,
            title_suffix="average home-cage session",
            fname_tag="avghcnorm",
        )

print()
print("=== Rats with NO usable HC baseline at all (every HC day was 0) ===")
any_missing = False
for name, no_baseline in all_no_baseline:
    for nb in no_baseline:
        any_missing = True
        print(
            f"  [{name}] rat {nb['rat']}: {nb['n_hc_days']} HC days total, "
            f"all had 0 co-occurrences -- no training days normalized for this rat"
        )
if not any_missing:
    print("  none -- every rat had at least one usable HC day")